
# BioRAG-X — 02 Ingestion & Provenance

## From forensic dataset → canonical biomedical knowledge layer

### Purpose

Notebook 01 answered **"What is in the dataset?"**

This notebook answers:

> **"How do we turn those raw records into a trustworthy, reproducible, versioned knowledge layer that every later BioRAG-X component can safely consume?"**

This is the foundation for:

- adaptive chunking
- BM25
- dense embeddings
- ANN
- GraphRAG
- PageIndex
- agentic retrieval
- evaluation
- production observability

We will build the ingestion layer **before** building any retrieval index.

---

## Learning goals

By the end of this notebook you should understand:

1. Raw data vs canonical data
2. Schema contracts
3. Data normalization
4. Stable identifiers
5. Content hashing
6. Provenance and lineage
7. Dataset/version fingerprints
8. Deduplication without losing provenance
9. Metadata enrichment
10. QA ↔ evidence relationships
11. Validation and data-quality gates
12. Immutable raw layer vs derived layer
13. Why ingestion decisions affect retrieval quality later

> **Engineering principle:** downstream retrieval quality can never be more trustworthy than the data layer it retrieves from.



## Research / engineering basis

BioRAG-X treats data as a first-class artifact.

The Hugging Face RAG datasets project describes `rag-mini-bioasq` as a corpus already split into passages with a test dataset containing questions, short answers and relevant passage IDs. That relationship is exactly what we need to preserve as provenance rather than flattening everything into plain text. citeturn863578search13

Hugging Face also recommends documenting dataset contents, creation context and important usage considerations through dataset cards. citeturn863578search0turn863578search1

For production reproducibility, dataset versioning and experiment state should be tracked as first-class artifacts; DVC's reproducibility guidance is a useful reference for this principle. citeturn863578search2turn863578search3

**Important:** this notebook does not require DVC. We first create the internal provenance contract; DVC/Git-LFS/object storage can be added later as the deployment environment requires.



# 1. Mental model: three data layers

We will intentionally separate:

### Layer A — RAW
Exactly what came from the upstream dataset.

Never mutate it.

### Layer B — CANONICAL
Normalized BioRAG-X records with stable IDs, normalized text, metadata and provenance.

### Layer C — DERIVED
Anything created later:

- chunks
- embeddings
- ANN indexes
- BM25 indexes
- graph nodes/edges
- PageIndex structures
- benchmark artifacts

Why?

Because if a retrieval experiment gives a surprising result, we must be able to trace:

`answer → evidence → chunk → canonical passage → raw source`

and reproduce that exact lineage.


In [1]:

from pathlib import Path
from datetime import datetime, timezone
from ast import literal_eval
from collections import Counter
import hashlib
import platform
import re
import sys
import unicodedata

import numpy as np
import pandas as pd

from datasets import load_dataset

SEED = 42
DATASET_ID = "rag-datasets/rag-mini-bioasq"
QA_CONFIG = "question-answer-passages"
CORPUS_CONFIG = "text-corpus"

PROJECT_ROOT = Path(".")
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "02_ingestion_and_provenance"
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CANONICAL_DIR = PROJECT_ROOT / "data" / "canonical"

for p in [ARTIFACT_DIR, RAW_DIR, CANONICAL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Project root:", PROJECT_ROOT.resolve())
print("Artifact dir:", ARTIFACT_DIR.resolve())


Python: 3.11.7
Platform: Windows-10-10.0.26200-SP0
Project root: C:\Users\10742161\OneDrive - LTIMindtree\Desktop\BioRAG-X\Notebooks
Artifact dir: C:\Users\10742161\OneDrive - LTIMindtree\Desktop\BioRAG-X\Notebooks\artifacts\02_ingestion_and_provenance


## 2. Re-load the upstream data

In [2]:

def first_split(dataset_dict):
    names = list(dataset_dict.keys())
    if not names:
        raise ValueError("Dataset configuration has no splits.")
    return names[0]

qa_raw_dict = load_dataset(DATASET_ID, QA_CONFIG)
corpus_raw_dict = load_dataset(DATASET_ID, CORPUS_CONFIG)

qa_split = "test" if "test" in qa_raw_dict else first_split(qa_raw_dict)
corpus_split = "passages" if "passages" in corpus_raw_dict else first_split(corpus_raw_dict)

qa = qa_raw_dict[qa_split].to_pandas()
corpus = corpus_raw_dict[corpus_split].to_pandas()

print(f"QA: {QA_CONFIG}/{qa_split} -> {len(qa):,} rows")
print(f"Corpus: {CORPUS_CONFIG}/{corpus_split} -> {len(corpus):,} rows")
print("QA columns:", list(qa.columns))
print("Corpus columns:", list(corpus.columns))


QA: question-answer-passages/test -> 4,719 rows
Corpus: text-corpus/passages -> 40,221 rows
QA columns: ['question', 'answer', 'relevant_passage_ids', 'id']
Corpus columns: ['passage', 'id']


## 3. Freeze a source fingerprint before transforming anything


### Why a fingerprint?

Suppose six months from now the upstream dataset changes.

If the code still says `rag-mini-bioasq`, that is not enough to prove that the same data was used.

We therefore create deterministic fingerprints from:

- dataset/config/split identity
- row counts
- ordered column names
- schema representation
- serialized raw content hashes

This creates an ingestion-time identity for the source snapshot.


In [3]:

def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()

def stable_json_hash(obj) -> str:
    payload = __import__('json').dumps(obj, sort_keys=True, ensure_ascii=False, default=str).encode('utf-8')
    return sha256_bytes(payload)

def frame_schema_signature(frame: pd.DataFrame) -> dict:
    return {
        "columns": list(frame.columns),
        "dtypes": {c: str(frame[c].dtype) for c in frame.columns},
        "row_count": len(frame),
    }

qa_schema_signature = frame_schema_signature(qa)
corpus_schema_signature = frame_schema_signature(corpus)

print("QA schema hash:", stable_json_hash(qa_schema_signature))
print("Corpus schema hash:", stable_json_hash(corpus_schema_signature))


QA schema hash: f9cb02311a14ad2ba4cf7e16aec1c5da217b6d50b480a6219a32434ce75b1e1c
Corpus schema hash: e26ba18abc1efd1c302d3af93168949f9f492e53ed2b0bc77c312b003dac8e49


In [4]:

def dataframe_record_hashes(frame: pd.DataFrame):
    cols = list(frame.columns)
    hashes = []
    for row in frame[cols].itertuples(index=False, name=None):
        record = {c: row[i] for i, c in enumerate(cols)}
        hashes.append(stable_json_hash(record))
    return hashes

qa_raw_record_hashes = dataframe_record_hashes(qa)
corpus_raw_record_hashes = dataframe_record_hashes(corpus)

source_manifest = {
    "dataset_id": DATASET_ID,
    "qa": {
        "config": QA_CONFIG,
        "split": qa_split,
        "rows": len(qa),
        "schema_hash": stable_json_hash(qa_schema_signature),
        "ordered_records_hash": stable_json_hash(qa_raw_record_hashes),
    },
    "corpus": {
        "config": CORPUS_CONFIG,
        "split": corpus_split,
        "rows": len(corpus),
        "schema_hash": stable_json_hash(corpus_schema_signature),
        "ordered_records_hash": stable_json_hash(corpus_raw_record_hashes),
    },
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}

source_snapshot_id = stable_json_hash({"dataset_id": DATASET_ID, "qa": source_manifest["qa"], "corpus": source_manifest["corpus"]})[:20]
source_manifest["source_snapshot_id"] = source_snapshot_id

print("Source snapshot ID:", source_snapshot_id)
print(__import__('json').dumps(source_manifest, indent=2)[:2500])


Source snapshot ID: 43ef01077ff18157b1d1
{
  "dataset_id": "rag-datasets/rag-mini-bioasq",
  "qa": {
    "config": "question-answer-passages",
    "split": "test",
    "rows": 4719,
    "schema_hash": "f9cb02311a14ad2ba4cf7e16aec1c5da217b6d50b480a6219a32434ce75b1e1c",
    "ordered_records_hash": "41709cf6350ba138f34a8e587a13c34207a9619fbb7a0574239a91d94a89d023"
  },
  "corpus": {
    "config": "text-corpus",
    "split": "passages",
    "rows": 40221,
    "schema_hash": "e26ba18abc1efd1c302d3af93168949f9f492e53ed2b0bc77c312b003dac8e49",
    "ordered_records_hash": "d3c0e37d655fe71afa7f42e76149a2ca7e030eadc6e2f84a307e9fc9c6da384a"
  },
  "created_at_utc": "2026-09-15T09:20:50.521855+00:00",
  "source_snapshot_id": "43ef01077ff18157b1d1"
}


## 4. Preserve the raw layer


We don't want canonicalization code to become our only copy of the source.

For reproducibility, we persist a lightweight manifest plus a raw snapshot where practical.

**Production note:** for large datasets, the actual raw bytes should be stored in immutable object storage or a data-versioning system rather than committed into Git. The notebook establishes the local contract first.


In [5]:

source_manifest_path = ARTIFACT_DIR / "source_manifest.json"
source_manifest_path.write_text(
    __import__('json').dumps(source_manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

qa_raw_path = RAW_DIR / f"qa_{source_snapshot_id}.parquet"
corpus_raw_path = RAW_DIR / f"corpus_{source_snapshot_id}.parquet"
qa.to_parquet(qa_raw_path, index=False)
corpus.to_parquet(corpus_raw_path, index=False)

print("Saved raw:")
print(" -", qa_raw_path)
print(" -", corpus_raw_path)
print(" -", source_manifest_path)


Saved raw:
 - data\raw\qa_43ef01077ff18157b1d1.parquet
 - data\raw\corpus_43ef01077ff18157b1d1.parquet
 - artifacts\02_ingestion_and_provenance\source_manifest.json


## 5. Canonical text normalization


### Important rule

Normalization must be **loss-aware**.

We want to remove accidental formatting noise, but we must not destroy biomedical meaning.

Safe examples:

- Unicode normalization
- line-ending normalization
- repeated whitespace
- surrounding whitespace

Potentially dangerous operations that we will **not** blindly do:

- stemming
- aggressive stopword removal
- lowercasing the stored source text
- punctuation deletion
- stripping symbols such as `+`, `-`, `/`, `%`
- altering Greek letters
- rewriting biomedical abbreviations

The canonical record therefore keeps:

`raw_text` + `normalized_text`

The raw form remains the provenance anchor.


In [6]:

def normalize_text(text: str) -> str:
    if text is None:
        return ""
    text = str(text)
    text = unicodedata.normalize("NFC", text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\t", " ")
    text = re.sub(r"[ ]{2,}", " ", text)
    text = "\n".join(line.strip() for line in text.split("\n"))
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

# Upstream rag-mini-bioasq stores missing passages as the LITERAL string 'nan'
# (not a true null), so fillna() does not catch them. We treat these sentinels
# as empty content rather than as real biomedical text.
MISSING_SENTINELS = {"", "nan", "none", "null", "n/a", "na"}

def is_empty_text(text) -> bool:
    """True if the value is missing, a float NaN, or a known missing sentinel string."""
    if text is None:
        return True
    if isinstance(text, float) and np.isnan(text):
        return True
    return str(text).strip().lower() in MISSING_SENTINELS

toy = "BRCA1\r\n\r\n  inhibits\tDNA repair.  "
print("RAW :", repr(toy))
print("NORM:", repr(normalize_text(toy)))
print("is_empty('nan'):", is_empty_text("nan"), "| is_empty(toy):", is_empty_text(toy))


RAW : 'BRCA1\r\n\r\n  inhibits\tDNA repair.  '
NORM: 'BRCA1\n\ninhibits DNA repair.'
is_empty('nan'): True | is_empty(toy): False


In [7]:

corpus_id_col = next((c for c in ["id", "passage_id", "document_id"] if c in corpus.columns), None)
text_col = next((c for c in ["text", "passage", "content", "document"] if c in corpus.columns), None)
if corpus_id_col is None:
    raise ValueError(f"Cannot identify corpus ID column: {list(corpus.columns)}")
if text_col is None:
    raise ValueError(f"Cannot identify corpus text column: {list(corpus.columns)}")

canonical_corpus = corpus.copy()
canonical_corpus["source_dataset"] = DATASET_ID
canonical_corpus["source_config"] = CORPUS_CONFIG
canonical_corpus["source_split"] = corpus_split
canonical_corpus["source_row_id"] = np.arange(len(canonical_corpus), dtype=np.int64)
canonical_corpus["source_passage_id"] = canonical_corpus[corpus_id_col].astype(str)
# Map missing sentinels (true NaN OR the literal string 'nan') to genuinely empty text.
_raw_text_series = canonical_corpus[text_col]
canonical_corpus["is_empty"] = _raw_text_series.map(is_empty_text)
canonical_corpus["raw_text"] = np.where(canonical_corpus["is_empty"], "", _raw_text_series.fillna("").astype(str))
canonical_corpus["is_usable"] = ~canonical_corpus["is_empty"]
canonical_corpus["normalized_text"] = canonical_corpus["raw_text"].map(normalize_text)
canonical_corpus["raw_text_sha256"] = canonical_corpus["raw_text"].map(lambda x: sha256_bytes(x.encode("utf-8")))
canonical_corpus["normalized_text_sha256"] = canonical_corpus["normalized_text"].map(lambda x: sha256_bytes(x.encode("utf-8")))
canonical_corpus["canonical_passage_id"] = canonical_corpus["source_passage_id"].map(lambda x: f"BIOP-{x}")
canonical_corpus["pipeline_version"] = "ingestion_v1"
canonical_corpus["source_snapshot_id"] = source_snapshot_id

display(canonical_corpus[["canonical_passage_id","source_passage_id","is_empty","raw_text","normalized_text"]].head(3))
print(f"\nEmpty/missing passages detected: {int(canonical_corpus['is_empty'].sum()):,} "
      f"({canonical_corpus['is_empty'].mean():.1%} of corpus)")
print(f"Usable passages: {int(canonical_corpus['is_usable'].sum()):,}")


,canonical_passage_id,source_passage_id,is_empty,raw_text,normalized_text
0,BIOP-9797,9797,False,New data on viruses isolated from patients wit...,New data on viruses isolated from patients wit...
1,BIOP-11906,11906,False,We describe an improved method for detecting d...,We describe an improved method for detecting d...
2,BIOP-16083,16083,False,We have studied the effects of curare on respo...,We have studied the effects of curare on respo...



Empty/missing passages detected: 12,220 (30.4% of corpus)
Usable passages: 28,001


## 6. Canonical question records


The important relationship is:

`question → gold_passage_ids → canonical_passage_id`

We retain both the upstream ID and our stable canonical ID.


In [8]:

def parse_passage_ids(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    if isinstance(value, (list, tuple, set, np.ndarray)):
        raw = list(value)
    elif isinstance(value, str):
        s = value.strip()
        if not s:
            return []
        try:
            parsed = literal_eval(s)
            raw = list(parsed) if isinstance(parsed, (list, tuple, set)) else [parsed]
        except (ValueError, SyntaxError):
            raw = re.findall(r"\d+", s)
    else:
        raw = [value]
    out, seen = [], set()
    for x in raw:
        sx = str(x).strip()
        if sx not in seen:
            seen.add(sx)
            out.append(sx)
    return out

required_qa_cols = {"question", "answer", "relevant_passage_ids", "id"}
missing = required_qa_cols - set(qa.columns)
if missing:
    raise ValueError(f"QA schema missing: {sorted(missing)}")

canonical_qa = qa.copy()
canonical_qa["source_dataset"] = DATASET_ID
canonical_qa["source_config"] = QA_CONFIG
canonical_qa["source_split"] = qa_split
canonical_qa["source_row_id"] = np.arange(len(canonical_qa), dtype=np.int64)
canonical_qa["source_question_id"] = canonical_qa["id"].astype(str)
canonical_qa["question"] = canonical_qa["question"].fillna("").astype(str)
canonical_qa["answer"] = canonical_qa["answer"].fillna("").astype(str)
canonical_qa["gold_source_passage_ids"] = canonical_qa["relevant_passage_ids"].map(parse_passage_ids)
canonical_qa["canonical_question_id"] = canonical_qa["source_question_id"].map(lambda x: f"BIOQ-{x}")
canonical_qa["gold_canonical_passage_ids"] = canonical_qa["gold_source_passage_ids"].map(lambda ids: [f"BIOP-{x}" for x in ids])
canonical_qa["question_sha256"] = canonical_qa["question"].map(lambda x: sha256_bytes(x.encode("utf-8")))
canonical_qa["answer_sha256"] = canonical_qa["answer"].map(lambda x: sha256_bytes(x.encode("utf-8")))
canonical_qa["pipeline_version"] = "ingestion_v1"
canonical_qa["source_snapshot_id"] = source_snapshot_id

display(canonical_qa[["canonical_question_id","question","answer","gold_source_passage_ids","gold_canonical_passage_ids"]].head(3))


,canonical_question_id,question,answer,gold_source_passage_ids,gold_canonical_passage_ids
0,BIOQ-0,Is Hirschsprung disease a mendelian or a multi...,"Coding sequence mutations in RET, GDNF, EDNRB,...","[20598273, 6650562, 15829955, 15617541, 230011...","[BIOP-20598273, BIOP-6650562, BIOP-15829955, B..."
1,BIOQ-1,List signaling molecules (ligands) that intera...,The 7 known EGFR ligands are: epidermal growt...,"[23821377, 24323361, 23382875, 22247333, 23787...","[BIOP-23821377, BIOP-24323361, BIOP-23382875, ..."
2,BIOQ-2,Is the protein Papilin secreted?,"Yes, papilin is a secreted protein","[21784067, 19297413, 15094122, 7515725, 332004...","[BIOP-21784067, BIOP-19297413, BIOP-15094122, ..."


## 7. Build an explicit provenance ledger


A **provenance ledger** answers:

> Where did this record come from, and what transformation produced it?

For a passage:

```text
HF dataset
   ↓
text-corpus / passages
   ↓
source row
   ↓
normalization
   ↓
canonical passage BIOP-...
```

Later it can continue:

```text
canonical passage
   ↓
semantic chunk
   ↓
embedding
   ↓
ANN result
   ↓
evidence
   ↓
claim
```

This is the backbone of traceability.


In [9]:

provenance_passages = canonical_corpus[[
    "canonical_passage_id","source_dataset","source_config","source_split",
    "source_row_id","source_passage_id","raw_text_sha256","normalized_text_sha256",
    "source_snapshot_id","pipeline_version"
]].copy()
provenance_passages["transformation"] = "source_load -> unicode_normalization -> whitespace_normalization -> canonical_id_assignment"

provenance_questions = canonical_qa[[
    "canonical_question_id","source_dataset","source_config","source_split",
    "source_row_id","source_question_id","question_sha256","answer_sha256",
    "source_snapshot_id","pipeline_version"
]].copy()
provenance_questions["transformation"] = "source_load -> ID_normalization -> passage_reference_normalization"

print("Passage provenance rows:", len(provenance_passages))
print("Question provenance rows:", len(provenance_questions))


Passage provenance rows: 40221
Question provenance rows: 4719


## 8. Join QA to canonical evidence

In [10]:

passage_id_set = set(canonical_corpus["canonical_passage_id"])
canonical_qa["missing_gold_canonical_ids"] = canonical_qa["gold_canonical_passage_ids"].map(lambda ids: [x for x in ids if x not in passage_id_set])
canonical_qa["gold_evidence_integrity_ok"] = canonical_qa["missing_gold_canonical_ids"].str.len().eq(0)

integrity_rate = canonical_qa["gold_evidence_integrity_ok"].mean()
print(f"Gold evidence integrity (IDs resolve): {integrity_rate:.2%}")
if integrity_rate < 1.0:
    display(canonical_qa.loc[~canonical_qa["gold_evidence_integrity_ok"], ["canonical_question_id","question","gold_canonical_passage_ids","missing_gold_canonical_ids"]].head(20))
else:
    print("All gold evidence references successfully resolve to canonical passage IDs.")

# A resolvable ID is not the same as usable evidence: an ID may point to an EMPTY passage.
empty_passage_ids = set(canonical_corpus.loc[canonical_corpus["is_empty"], "canonical_passage_id"])
canonical_qa["empty_gold_canonical_ids"] = canonical_qa["gold_canonical_passage_ids"].map(lambda ids: [x for x in ids if x in empty_passage_ids])
canonical_qa["usable_gold_canonical_ids"] = canonical_qa["gold_canonical_passage_ids"].map(lambda ids: [x for x in ids if x not in empty_passage_ids])
canonical_qa["has_usable_gold_evidence"] = canonical_qa["usable_gold_canonical_ids"].str.len().gt(0)

total_refs = int(canonical_qa["gold_canonical_passage_ids"].str.len().sum())
empty_refs = int(canonical_qa["empty_gold_canonical_ids"].str.len().sum())
no_usable = int((~canonical_qa["has_usable_gold_evidence"]).sum())
print(f"\nGold references pointing to EMPTY passages: {empty_refs:,} of {total_refs:,} ({empty_refs/max(1,total_refs):.1%})")
print(f"Questions with NO usable gold evidence at all: {no_usable:,} of {len(canonical_qa):,} ({no_usable/len(canonical_qa):.1%})")


Gold evidence integrity (IDs resolve): 100.00%
All gold evidence references successfully resolve to canonical passage IDs.

Gold references pointing to EMPTY passages: 12,925 of 42,608 (30.3%)
Questions with NO usable gold evidence at all: 332 of 4,719 (7.0%)


## 9. Deduplication: remove redundancy without destroying lineage


There are two very different operations:

### Exact duplicate
Same normalized text.

### Near duplicate
Texts differ slightly but may contain the same evidence.

We will **not** aggressively remove near duplicates in this notebook.

For now:

- identify exact duplicates
- assign a `duplicate_group_id`
- retain all source records
- let later retrieval experiments decide whether deduplication is useful

This is safer than silently deleting evidence.


In [11]:

dup_group = canonical_corpus.groupby("normalized_text_sha256", sort=False).ngroup().astype(np.int64)
canonical_corpus["exact_duplicate_group_id"] = dup_group.map(lambda x: f"DG-{x:08d}")
canonical_corpus["is_exact_duplicate"] = canonical_corpus.duplicated(subset=["normalized_text_sha256"], keep=False)

duplicate_summary = (canonical_corpus.groupby("exact_duplicate_group_id").agg(row_count=("canonical_passage_id","size"), source_ids=("source_passage_id",list)).query("row_count > 1").sort_values("row_count", ascending=False))
print("Exact duplicate groups:", len(duplicate_summary))
display(duplicate_summary.head(20))


Exact duplicate groups: 6


,row_count,source_ids
exact_duplicate_group_id,,
DG-00000011,12220,"[97949, 98518, 100785, 117628, 125891, 227209,..."
DG-00010378,24,"[20007090, 20639591, 21952424, 22550943, 23104..."
DG-00012261,2,"[21429248, 21549021]"
DG-00013283,2,"[21975940, 23255150]"
DG-00015132,2,"[22875912, 23233565]"
DG-00022970,2,"[27568285, 27697436]"


## 10. Metadata enrichment without hallucinating


Metadata should be computed from the actual record whenever possible.

We add deterministic structural metadata now:

- character count
- word count
- sentence count (lightweight heuristic)
- line count
- digit ratio
- uppercase ratio
- URL count
- PMID/DOI-like reference patterns

We intentionally do **not** infer biomedical entities in this notebook.

Entity extraction is a separate experiment because its model choice and errors must be evaluated independently.


In [12]:

SENTENCE_RE = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9])")

def sentence_count(text: str) -> int:
    text = text.strip()
    if not text:
        return 0
    return max(1, len(SENTENCE_RE.split(text)))

def structural_features(text: str) -> dict:
    chars = len(text)
    digits = sum(ch.isdigit() for ch in text)
    uppercase = sum(ch.isupper() for ch in text if ch.isalpha())
    alpha = sum(ch.isalpha() for ch in text)
    return {
        "char_count": chars,
        "word_count": len(text.split()),
        "sentence_count": sentence_count(text),
        "line_count": text.count("\n") + (1 if text else 0),
        "digit_ratio": digits / max(1, chars),
        "uppercase_alpha_ratio": uppercase / max(1, alpha),
        "url_count": len(re.findall(r"https?://\S+|www\.\S+", text)),
        "pmid_like_count": len(re.findall(r"\bPMID\s*:?\s*\d+\b", text, flags=re.I)),
        "doi_like_count": len(re.findall(r"\b10\.\d{4,9}/[-._;()/:A-Z0-9]+\b", text, flags=re.I)),
    }

features = canonical_corpus["normalized_text"].map(structural_features).apply(pd.Series)
canonical_corpus = pd.concat([canonical_corpus.reset_index(drop=True), features.reset_index(drop=True)], axis=1)

display(canonical_corpus[["canonical_passage_id","char_count","word_count","sentence_count","digit_ratio","pmid_like_count","doi_like_count"]].head(5))


,canonical_passage_id,char_count,word_count,sentence_count,digit_ratio,pmid_like_count,doi_like_count
0,BIOP-9797,355.0,49.0,4.0,0.000000,0.0,0.0
1,BIOP-11906,445.0,62.0,3.0,0.004494,0.0,0.0
2,BIOP-16083,1390.0,202.0,9.0,0.004317,0.0,0.0
3,BIOP-23188,810.0,110.0,3.0,0.017284,0.0,0.0
4,BIOP-23469,1466.0,207.0,8.0,0.036153,0.0,0.0


## 10b. Document profiler (deterministic biomedical-signal densities)

Before any chunking or retrieval decision, we profile each **usable** passage with deterministic signals that later stages use to reason about *how* a passage should be represented and retrieved:

- **acronym density** (BRCA1, EGFR, DNA) — high values favour exact/lexical (BM25) retrieval
- **hyphenated-term density** (TNF-alpha, anti-TNF) — tokenization-sensitive biomedical terms
- **negation-cue density** (no, not, without, absence) — meaning-critical in medicine; must not be split away
- **reference density** (PMID/DOI/[17,18]) — structural citation markers
- **digit-token density** and **avg sentence length** — structural profile signals

These are computed **only for non-empty passages** (the `nan` passages are excluded). This is NOT biomedical NER — true entity/relation extraction needs an ML model whose errors must be evaluated independently, so it is **deferred to a dedicated later notebook**. Here we only record deterministic, reproducible signals.

In [13]:
GREEK_CHARS = set("αβγδεζηθικλμνξοπρσςτυφχψωΑΒΓΔΕΖΗΘΙΚΛΜΝΞΟΠΡΣΤΥΦΧΨΩ")
ACRONYM_RE = re.compile(r"\b[A-Z][A-Z0-9]{1,}\b")
HYPHEN_TERM_RE = re.compile(r"\b\w+-\w+(?:-\w+)*\b")
DIGIT_TOKEN_RE = re.compile(r"\b\w*\d\w*\b")
NEGATION_RE = re.compile(r"\b(no|not|non|without|absence|absent|neither|nor|never|cannot|lack|lacking|fail(?:ed|s)?)\b", flags=re.I)
REFERENCE_RE = re.compile(r"\bPMID\s*:?\s*\d+\b|\b10\.\d{4,9}/[-._;()/:A-Z0-9]+\b|\[\d+(?:\s*,\s*\d+)*\]", flags=re.I)

def document_profile(text: str) -> dict:
    tokens = text.split()
    n_tok = max(1, len(tokens))
    n_sent = max(1, sentence_count(text))
    acronyms = ACRONYM_RE.findall(text)
    hyphenated = HYPHEN_TERM_RE.findall(text)
    digits = DIGIT_TOKEN_RE.findall(text)
    negations = NEGATION_RE.findall(text)
    references = REFERENCE_RE.findall(text)
    greek = sum(1 for ch in text if ch in GREEK_CHARS)
    return {
        "acronym_density": len(acronyms) / n_tok,
        "hyphenated_density": len(hyphenated) / n_tok,
        "digit_token_density": len(digits) / n_tok,
        "negation_cue_count": len(negations),
        "negation_density": len(negations) / n_sent,
        "reference_marker_count": len(references),
        "greek_char_count": greek,
        "avg_sentence_length_words": len(tokens) / n_sent,
    }

# Profile ONLY usable passages; empty passages get null-profile (0.0) so the schema stays uniform.
profile_cols = ["acronym_density","hyphenated_density","digit_token_density","negation_cue_count",
                "negation_density","reference_marker_count","greek_char_count","avg_sentence_length_words"]

_profiles = canonical_corpus.apply(
    lambda r: document_profile(r["normalized_text"]) if r["is_usable"] else {c: 0.0 for c in profile_cols},
    axis=1,
).apply(pd.Series)
canonical_corpus = pd.concat([canonical_corpus.reset_index(drop=True), _profiles.reset_index(drop=True)], axis=1)

_usable = canonical_corpus[canonical_corpus["is_usable"]]
document_profiler_summary = {
    "usable_passages_profiled": int(len(_usable)),
    "mean_acronym_density": float(_usable["acronym_density"].mean()),
    "mean_hyphenated_density": float(_usable["hyphenated_density"].mean()),
    "mean_negation_density": float(_usable["negation_density"].mean()),
    "passages_with_acronym_pct": float((_usable["acronym_density"] > 0).mean() * 100),
    "passages_with_negation_pct": float((_usable["negation_cue_count"] > 0).mean() * 100),
    "passages_with_reference_pct": float((_usable["reference_marker_count"] > 0).mean() * 100),
    "passages_with_greek_pct": float((_usable["greek_char_count"] > 0).mean() * 100),
}
print("Document profiler summary (usable passages only):")
for k, v in document_profiler_summary.items():
    print(f"  {k}: {round(v, 4) if isinstance(v, float) else v}")
print("\nNOTE: ML biomedical entity/relation extraction is DEFERRED to a dedicated later notebook.")
display(_usable[["canonical_passage_id"] + profile_cols].head(5))


Document profiler summary (usable passages only):
  usable_passages_profiled: 28001
  mean_acronym_density: 0.0502
  mean_hyphenated_density: 0.0292
  mean_negation_density: 0.1274
  passages_with_acronym_pct: 88.9183
  passages_with_negation_pct: 58.5265
  passages_with_reference_pct: 0.4964
  passages_with_greek_pct: 8.864

NOTE: ML biomedical entity/relation extraction is DEFERRED to a dedicated later notebook.


,canonical_passage_id,acronym_density,hyphenated_density,digit_token_density,negation_cue_count,negation_density,reference_marker_count,greek_char_count,avg_sentence_length_words
0,BIOP-9797,0.000000,0.020408,0.000000,0.0,0.000000,0.0,0.0,12.250000
1,BIOP-11906,0.016129,0.064516,0.032258,0.0,0.000000,0.0,0.0,20.666667
2,BIOP-16083,0.009901,0.019802,0.019802,2.0,0.222222,0.0,0.0,22.444444
3,BIOP-23188,0.054545,0.045455,0.090909,1.0,0.333333,0.0,0.0,36.666667
4,BIOP-23469,0.000000,0.038647,0.101449,0.0,0.000000,0.0,0.0,25.875000


## 11. Add retrieval-safe metadata


The canonical layer should be **retrieval-neutral**.

We store useful metadata now, but avoid encoding a retrieval strategy into the record itself.

For example, BM25 can use `normalized_text`, dense retrieval can use the same field, semantic chunking can use sentence boundaries, GraphRAG can attach entities/relations, and PageIndex can use hierarchy/provenance.

The canonical layer is the **shared contract**, not an index.


In [14]:

canonical_corpus["retrieval_text"] = canonical_corpus["normalized_text"]

canonical_corpus["metadata_json"] = canonical_corpus.apply(
    lambda r: __import__('json').dumps({
        "source_dataset": r["source_dataset"],
        "source_config": r["source_config"],
        "source_split": r["source_split"],
        "source_row_id": int(r["source_row_id"]),
        "source_passage_id": str(r["source_passage_id"]),
        "source_snapshot_id": r["source_snapshot_id"],
        "pipeline_version": r["pipeline_version"],
        "word_count": int(r["word_count"]),
        "sentence_count": int(r["sentence_count"]),
        "exact_duplicate_group_id": r["exact_duplicate_group_id"],
        "is_empty": bool(r["is_empty"]),
    }, sort_keys=True), axis=1
)

display(canonical_corpus[["canonical_passage_id","retrieval_text","metadata_json"]].head(2))


,canonical_passage_id,retrieval_text,metadata_json
0,BIOP-9797,New data on viruses isolated from patients wit...,"{""exact_duplicate_group_id"": ""DG-00000000"", ""i..."
1,BIOP-11906,We describe an improved method for detecting d...,"{""exact_duplicate_group_id"": ""DG-00000001"", ""i..."


## 12. Data-quality gates


Before canonical data is allowed into downstream indexes, it should pass explicit gates.

### Hard failures

- duplicate canonical IDs
- missing canonical text
- missing source provenance
- broken gold references
- invalid schema

### Soft warnings

- unusually short text
- unusually long text
- repeated normalized passages
- suspicious characters
- extreme digit/uppercase ratios

The production idea is:

> **Fail early, before indexing.**


In [15]:

def run_ingestion_quality_gate(qa_df, corpus_df):
    errors = []
    warnings = []
    metrics = {}
    if not corpus_df["canonical_passage_id"].is_unique:
        errors.append("Canonical passage IDs are not unique.")
    # An UNEXPECTED empty is a passage with empty text that was NOT flagged is_empty.
    unexpected_empty = int((corpus_df["retrieval_text"].str.len().eq(0) & ~corpus_df["is_empty"]).sum())
    if unexpected_empty:
        errors.append(f"{unexpected_empty} passages have empty retrieval_text but are not flagged is_empty.")
    required_passage_fields = ["canonical_passage_id","raw_text","normalized_text","retrieval_text","is_empty","is_usable","source_snapshot_id","pipeline_version"]
    for col in required_passage_fields:
        if col not in corpus_df.columns:
            errors.append(f"Missing canonical passage field: {col}")
    if not qa_df["canonical_question_id"].is_unique:
        errors.append("Canonical question IDs are not unique.")
    if not qa_df["gold_evidence_integrity_ok"].all():
        errors.append("At least one QA row references a non-existent canonical gold passage ID.")
    # First-class data-quality metrics for this corpus.
    n_empty = int(corpus_df["is_empty"].sum())
    n_usable = int(corpus_df["is_usable"].sum())
    metrics["empty_passages"] = n_empty
    metrics["empty_passage_rate"] = round(n_empty / max(1, len(corpus_df)), 4)
    metrics["usable_passages"] = n_usable
    metrics["questions_without_usable_gold"] = int((~qa_df["has_usable_gold_evidence"]).sum())
    metrics["empty_gold_references"] = int(qa_df["empty_gold_canonical_ids"].str.len().sum())
    # Soft warnings
    if n_empty:
        warnings.append(f"{n_empty} empty/missing passages ({metrics['empty_passage_rate']:.1%} of corpus) - excluded from usable evidence.")
    if metrics["questions_without_usable_gold"]:
        warnings.append(f"{metrics['questions_without_usable_gold']} questions have NO usable (non-empty) gold evidence.")
    very_short_usable = int(((corpus_df["word_count"] < 5) & corpus_df["is_usable"]).sum())
    if very_short_usable:
        warnings.append(f"{very_short_usable} USABLE passages still have <5 words.")
    return errors, warnings, metrics

errors, warnings, gate_metrics = run_ingestion_quality_gate(canonical_qa, canonical_corpus)
print("ERRORS:")
for e in errors: print(" -", e)
print("\nWARNINGS:")
for w in warnings: print(" -", w)
print("\nDATA-QUALITY METRICS:")
for k, v in gate_metrics.items(): print(f"  {k}: {v}")
if errors:
    raise RuntimeError("Canonical ingestion quality gate FAILED.")
print("\nCanonical ingestion quality gate PASSED.")


ERRORS:

WARNINGS:
 - 12220 empty/missing passages (30.4% of corpus) - excluded from usable evidence.
 - 332 questions have NO usable (non-empty) gold evidence.
 - 24 USABLE passages still have <5 words.

DATA-QUALITY METRICS:
  empty_passages: 12220
  empty_passage_rate: 0.3038
  usable_passages: 28001
  questions_without_usable_gold: 332
  empty_gold_references: 12925

Canonical ingestion quality gate PASSED.


## 13. Build the canonical evidence relationship table


Instead of embedding gold relationships inside a nested JSON field only, create an explicit relationship table:

`question_id ↔ passage_id`

This will later support:

- retrieval Recall@K
- MRR
- multi-passage coverage
- evidence-set evaluation
- benchmark analysis
- graph-style provenance


In [16]:

empty_passage_ids = set(canonical_corpus.loc[canonical_corpus["is_empty"], "canonical_passage_id"])
relationship_rows = []
for _, row in canonical_qa.iterrows():
    for rank, passage_id in enumerate(row["gold_canonical_passage_ids"], start=1):
        relationship_rows.append({
            "canonical_question_id": row["canonical_question_id"],
            "canonical_passage_id": passage_id,
            "gold_evidence_rank": rank,
            "evidence_is_empty": passage_id in empty_passage_ids,
            "source_snapshot_id": row["source_snapshot_id"],
            "relationship_type": "gold_relevant_evidence",
            "provenance": "BioASQ-derived upstream annotation",
        })
gold_relationships = pd.DataFrame(relationship_rows)
n_usable_rel = int((~gold_relationships["evidence_is_empty"]).sum())
print("Gold QA↔passage relationships:", len(gold_relationships))
print(f"  usable (non-empty evidence): {n_usable_rel:,}")
print(f"  pointing to empty passages : {len(gold_relationships) - n_usable_rel:,}")
display(gold_relationships.head(10))


Gold QA↔passage relationships: 42608
  usable (non-empty evidence): 29,683
  pointing to empty passages : 12,925


,canonical_question_id,canonical_passage_id,gold_evidence_rank,evidence_is_empty,source_snapshot_id,relationship_type,provenance
0,BIOQ-0,BIOP-20598273,1,False,43ef01077ff18157b1d1,gold_relevant_evidence,BioASQ-derived upstream annotation
1,BIOQ-0,BIOP-6650562,2,True,43ef01077ff18157b1d1,gold_relevant_evidence,BioASQ-derived upstream annotation
2,BIOQ-0,BIOP-15829955,3,True,43ef01077ff18157b1d1,gold_relevant_evidence,BioASQ-derived upstream annotation
3,BIOQ-0,BIOP-15617541,4,False,43ef01077ff18157b1d1,gold_relevant_evidence,BioASQ-derived upstream annotation
4,BIOQ-0,BIOP-23001136,5,False,43ef01077ff18157b1d1,gold_relevant_evidence,BioASQ-derived upstream annotation
5,BIOQ-0,BIOP-8896569,6,True,43ef01077ff18157b1d1,gold_relevant_evidence,BioASQ-derived upstream annotation
6,BIOQ-0,BIOP-21995290,7,True,43ef01077ff18157b1d1,gold_relevant_evidence,BioASQ-derived upstream annotation
7,BIOQ-0,BIOP-12239580,8,False,43ef01077ff18157b1d1,gold_relevant_evidence,BioASQ-derived upstream annotation
8,BIOQ-0,BIOP-15858239,9,True,43ef01077ff18157b1d1,gold_relevant_evidence,BioASQ-derived upstream annotation
9,BIOQ-1,BIOP-23821377,1,True,43ef01077ff18157b1d1,gold_relevant_evidence,BioASQ-derived upstream annotation


## 14. Create a human-auditable lineage example

In [17]:

example_q = canonical_qa.iloc[0]
print("QUESTION")
print(example_q["canonical_question_id"])
print(example_q["question"])
print("\nGOLD PASSAGES")
for pid in example_q["gold_canonical_passage_ids"]:
    match = canonical_corpus.loc[canonical_corpus["canonical_passage_id"] == pid]
    if match.empty:
        continue
    row = match.iloc[0]
    print(f"\n{pid}")
    print("source ID:", row["source_passage_id"])
    print("raw hash :", row["raw_text_sha256"][:16])
    print("norm hash:", row["normalized_text_sha256"][:16])
    print("text     :", row["normalized_text"][:500].replace("\n", " "))


QUESTION
BIOQ-0
Is Hirschsprung disease a mendelian or a multifactorial disorder?

GOLD PASSAGES

BIOP-20598273
source ID: 20598273
raw hash : fbe44c3331f1c640
norm hash: 782e9f758f553eb3
text     : The major gene for Hirschsprung disease (HSCR) encodes the receptor tyrosine kinase RET. In a study of 690 European- and 192 Chinese-descent probands and their parents or controls, we demonstrate the ubiquity of a >4-fold susceptibility from a C-->T allele (rs2435357: p = 3.9 x 10(-43) in European ancestry; p = 1.1 x 10(-21) in Chinese samples) that probably arose once within the intronic RET enhancer MCS+9.7. With in vitro assays, we now show that the T variant disrupts a SOX10 binding site wit

BIOP-6650562
source ID: 6650562
raw hash : e3b0c44298fc1c14
norm hash: e3b0c44298fc1c14
text     : 

BIOP-15829955
source ID: 15829955
raw hash : e3b0c44298fc1c14
norm hash: e3b0c44298fc1c14
text     : 

BIOP-15617541
source ID: 15617541
raw hash : 430dbe73f17bf027
norm hash: ba948697b9a8943c
text 


### Why this lineage view matters

Later the UI should let us perform this exact operation for any answer:

`answer claim → citation → evidence chunk → parent passage → canonical passage → raw source`

If we cannot explain where a record came from, it should not be considered production-grade evidence.


## 15. Dataset statistics after canonicalization

In [18]:

canonical_stats = pd.DataFrame({
    "metric": [
        "canonical QA rows",
        "canonical passage rows",
        "empty/missing passages",
        "usable passages",
        "unique normalized passage texts",
        "exact duplicate passage rows",
        "gold QA↔passage relationships",
        "gold relationships to empty passages",
        "QA rows with valid gold IDs",
        "QA rows with usable gold evidence",
        "median USABLE passage words",
        "p95 USABLE passage words",
        "median question words",
    ],
    "value": [
        len(canonical_qa),
        len(canonical_corpus),
        int(canonical_corpus["is_empty"].sum()),
        int(canonical_corpus["is_usable"].sum()),
        canonical_corpus["normalized_text_sha256"].nunique(),
        int(canonical_corpus["is_exact_duplicate"].sum()),
        len(gold_relationships),
        int(gold_relationships["evidence_is_empty"].sum()),
        int(canonical_qa["gold_evidence_integrity_ok"].sum()),
        int(canonical_qa["has_usable_gold_evidence"].sum()),
        float(canonical_corpus.loc[canonical_corpus["is_usable"], "word_count"].median()),
        float(canonical_corpus.loc[canonical_corpus["is_usable"], "word_count"].quantile(0.95)),
        float(canonical_qa["question"].str.split().str.len().median()),
    ],
})
display(canonical_stats)


,metric,value
0,canonical QA rows,4719.0
1,canonical passage rows,40221.0
2,empty/missing passages,12220.0
3,usable passages,28001.0
4,unique normalized passage texts,27975.0
5,exact duplicate passage rows,12252.0
6,gold QA↔passage relationships,42608.0
7,gold relationships to empty passages,12925.0
8,QA rows with valid gold IDs,4719.0
9,QA rows with usable gold evidence,4387.0


## 16. Persist the canonical knowledge layer


### Output contract

#### `passages`
One row per canonical source passage.

#### `questions`
One row per canonical question.

#### `gold_relationships`
One row per question ↔ gold passage relation.

The tables are deliberately simple and relational. This makes them easy to use from pandas, DuckDB, Polars, Spark, vector indexes, search engines and graph databases without changing the underlying contract.


In [19]:

passages_path = CANONICAL_DIR / "passages.parquet"
questions_path = CANONICAL_DIR / "questions.parquet"
gold_relationships_path = CANONICAL_DIR / "gold_relationships.parquet"
provenance_passages_path = CANONICAL_DIR / "provenance_passages.parquet"
provenance_questions_path = CANONICAL_DIR / "provenance_questions.parquet"

canonical_corpus.to_parquet(passages_path, index=False)
canonical_qa.to_parquet(questions_path, index=False)
gold_relationships.to_parquet(gold_relationships_path, index=False)
provenance_passages.to_parquet(provenance_passages_path, index=False)
provenance_questions.to_parquet(provenance_questions_path, index=False)

print("Canonical outputs:")
for p in [passages_path, questions_path, gold_relationships_path, provenance_passages_path, provenance_questions_path]:
    print(" -", p)


Canonical outputs:
 - data\canonical\passages.parquet
 - data\canonical\questions.parquet
 - data\canonical\gold_relationships.parquet
 - data\canonical\provenance_passages.parquet
 - data\canonical\provenance_questions.parquet


## 17. Write an ingestion manifest

In [20]:

canonical_manifest = {
    "project": "BioRAG-X",
    "pipeline": "02_ingestion_and_provenance",
    "pipeline_version": "ingestion_v1",
    "source_snapshot_id": source_snapshot_id,
    "source": {
        "dataset_id": DATASET_ID,
        "qa_config": QA_CONFIG,
        "qa_split": qa_split,
        "corpus_config": CORPUS_CONFIG,
        "corpus_split": corpus_split,
    },
    "outputs": {
        "passages": str(passages_path),
        "questions": str(questions_path),
        "gold_relationships": str(gold_relationships_path),
        "provenance_passages": str(provenance_passages_path),
        "provenance_questions": str(provenance_questions_path),
    },
    "row_counts": {
        "passages": len(canonical_corpus),
        "questions": len(canonical_qa),
        "gold_relationships": len(gold_relationships),
    },
    "quality_gate": {"status": "PASSED", "errors": errors, "warnings": warnings, "metrics": gate_metrics},
    "document_profiler": document_profiler_summary,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}
manifest_path = ARTIFACT_DIR / "ingestion_manifest.json"
manifest_path.write_text(__import__('json').dumps(canonical_manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print(manifest_path)


artifacts\02_ingestion_and_provenance\ingestion_manifest.json



# 18. What we learned

### Raw → Canonical

We now have:

`raw record → canonical record`

with:

- stable identity
- source identity
- content fingerprint
- transformation version
- source snapshot

### Canonical → Retrieval

Later notebooks must consume the canonical contract rather than reaching directly into the Hugging Face dataset.

### Provenance → Explainability

Every future retrieved item can be traced back to the source record.

### QA → Evidence

Gold question/passage relationships are represented explicitly and remain available for evaluation.

---

# Why we stop here

**We do not chunk, embed, build BM25, build HNSW, create the graph, or call an LLM in this notebook.**

Those are downstream transformations.

Keeping ingestion separate lets us answer:

> "Did retrieval fail because retrieval was bad, or because our data representation was bad?"

That separation is critical for scientific experimentation.



# 19. Handoff to Notebook 03

Notebook 03 will consume:

- `data/canonical/passages.parquet`
- `data/canonical/questions.parquet`
- `data/canonical/gold_relationships.parquet`
- provenance tables
- `artifacts/02_ingestion_and_provenance/ingestion_manifest.json`

Notebook 03 will begin **chunking experiments**, starting with:

1. fixed-size
2. recursive / structure-preserving

We will measure downstream impact before adding semantic, biomedical-aware, proposition, parent-child, late, query-adaptive, LLM-guided and agentic chunking.

That preserves controlled experimentation.



# Expected deliverables

After successful execution, this notebook should create:

```text
data/
├── raw/
│   ├── qa_<source_snapshot>.parquet
│   └── corpus_<source_snapshot>.parquet
│
└── canonical/
    ├── passages.parquet
    ├── questions.parquet
    ├── gold_relationships.parquet
    ├── provenance_passages.parquet
    └── provenance_questions.parquet

artifacts/
└── 02_ingestion_and_provenance/
    ├── source_manifest.json
    └── ingestion_manifest.json
```

The canonical Parquet files are the **contract for the rest of BioRAG-X**.
